In [ ]:
import json
import logging
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(message)s")
logger = logging.getLogger(__name__)


def load_embeddings(embedding_path: Path) -> Tuple[np.ndarray, str]:
    ALLOWED_FILES = {
        "nv_embed.npy",
        "jina_base_text_match.npy",
        "mnrl_full_data.npy",
        "gatortron_large.npy",
    }

    if embedding_path.suffix == ".npy":
        if embedding_path.name in ALLOWED_FILES:
            embeddings = np.load(embedding_path)
            model_name = embedding_path.stem
            return embeddings, model_name

    return None, None


def evaluate_fold(
    X_train: np.ndarray, X_test: np.ndarray, y_train: np.ndarray, y_test: np.ndarray
) -> Dict:
    """Evaluate a single fold for both tumor_type and survival metrics."""
    clf = RandomForestClassifier(
        n_estimators=1000, class_weight="balanced", random_state=7, n_jobs=-1
    )
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    y_pred_proba = clf.predict_proba(X_test)

    # Calculate all metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision, recall, f1_macro, _ = precision_recall_fscore_support(
        y_test, y_pred, average="macro", zero_division=0
    )

    metrics = {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_macro": f1_macro,
    }

    # Add AUC for all tasks using one-vs-rest for multiclass
    if len(np.unique(y_test)) == 2:
        auc = roc_auc_score(y_test, y_pred_proba[:, 1])
    else:
        auc = roc_auc_score(y_test, y_pred_proba, multi_class="ovr")
    metrics["aucroc"] = auc

    return metrics


def run_kfold_evaluation(
    embeddings: np.ndarray,
    labels: np.ndarray,
    n_splits: int = 5,
) -> Dict:
    """Run k-fold cross validation."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=7)
    fold_metrics = []

    for _, (train_idx, test_idx) in tqdm(
        enumerate(skf.split(embeddings, labels)),
        total=n_splits,
        desc="Running k-fold evaluation",
    ):
        X_train, X_test = embeddings[train_idx], embeddings[test_idx]
        y_train, y_test = labels[train_idx], labels[test_idx]

        metrics = evaluate_fold(X_train, X_test, y_train, y_test)
        fold_metrics.append(metrics)

    # Average metrics across folds
    avg_metrics = {}
    for metric in fold_metrics[0].keys():
        values = [fold[metric] for fold in fold_metrics]
        avg_metrics[metric] = np.mean(values)
        avg_metrics[f"{metric}_std"] = np.std(values)
        avg_metrics[f"{metric}_values"] = (
            values  # Store all values for statistical testing
        )

    return avg_metrics


def calculate_cohens_d(baseline: List[float], comparison: List[float]) -> float:
    """Calculate Cohen's d effect size."""
    n1, n2 = len(baseline), len(comparison)
    var1, var2 = np.var(baseline, ddof=1), np.var(comparison, ddof=1)

    # Pooled standard deviation
    pooled_se = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))

    # Cohen's d
    d = (np.mean(comparison) - np.mean(baseline)) / pooled_se
    return d


def calculate_significance(
    baseline_values: List[float], comparison_values: List[float]
) -> Tuple[float, str, float, str]:
    """Calculate statistical significance using multiple tests."""
    # Paired t-test
    t_stat, p_value = stats.ttest_rel(baseline_values, comparison_values)

    # Effect size
    effect_size = calculate_cohens_d(baseline_values, comparison_values)

    # Effect size interpretation
    if abs(effect_size) > 0.8:
        effect_marker = "‡"  # Large effect
    elif abs(effect_size) > 0.5:
        effect_marker = "†"  # Medium effect
    elif abs(effect_size) > 0.2:
        effect_marker = "§"  # Small effect
    else:
        effect_marker = ""

    # Statistical significance markers
    if p_value < 0.001:
        significance = "***"
    elif p_value < 0.01:
        significance = "**"
    elif p_value < 0.05:
        significance = "*"
    else:
        significance = ""

    return p_value, significance + effect_marker


def format_results_table(
    results: Dict[str, Dict], task: str, baseline_model: str = "jina_base_text_match"
) -> str:
    """Format results as a markdown table with significance testing."""
    # Define metric order and display names
    metric_order = ["accuracy", "precision", "recall", "f1_macro", "aucroc"]
    metric_display = {
        "accuracy": "Accuracy",
        "precision": "Precision",
        "recall": "Recall",
        "f1_macro": "F1-macro",
        "aucroc": "AUCROC",
    }

    metrics = [m for m in metric_order if m in next(iter(results.values()))]
    headers = ["Model"] + [f"{metric_display[m]} (%)" for m in metrics]

    table = ["|" + "|".join(headers) + "|"]
    table.append("|" + "|".join(["---"] * len(headers)) + "|")

    # Use first model as baseline if baseline_model not found
    if baseline_model not in results:
        baseline_model = list(results.keys())[0]
        logger.info(
            f"Baseline model '{baseline_model}' not found. Using '{baseline_model}' as baseline."
        )

    baseline_results = results[baseline_model]

    # Sort models to ensure baseline is first
    sorted_models = sorted(results.keys(), key=lambda x: (x != baseline_model, x))

    for model in sorted_models:
        model_results = results[model]
        row = [model]
        for metric in metrics:
            value = model_results[metric] * 100  # Convert to percentage
            std = model_results[f"{metric}_std"] * 100

            # Calculate significance if not baseline
            if model != baseline_model:
                _, sig = calculate_significance(
                    baseline_results[f"{metric}_values"],
                    model_results[f"{metric}_values"],
                )
            else:
                sig = ""

            row.append(f"{value:.1f} ± {std:.1f}{sig}")

        table.append("|" + "|".join(row) + "|")

    # Add legend
    table.append("|" + "|".join([""] * len(headers)) + "|")
    table.append("|" + "**Legend:**" + "|".join([""] * (len(headers) - 1)) + "|")
    table.append(
        "|" + "* p < 0.05, ** p < 0.01" + "|".join([""] * (len(headers) - 1)) + "|"
    )
    table.append(
        "|"
        + "§ small effect, † medium effect, ‡ large effect"
        + "|".join([""] * (len(headers) - 1))
        + "|"
    )

    return "\n".join(table)


def main():
    # Load labels
    tumor_df = pd.read_csv("data/eval/tumor_type_eval.csv")
    mgmt_df = pd.read_csv("data/eval/mgmt_eval.csv")
    os_eval = pd.read_csv("data/eval/os_eval.csv")
    mgmt_df["mgmt_label"] = (
        mgmt_df["mgmt"].notna() & (mgmt_df["mgmt"] == "positive")
    ).astype(int)

    # Create results directory
    results_dir = Path("results")
    results_dir.mkdir(exist_ok=True)

    # Evaluate embeddings for each task
    tasks = {
        "tumor_type": {
            "embeddings_dir": Path("embeddings/tumor_type"),
            "df": tumor_df,
            "label_col": "class_label",
            "id_col": "MRN",
        },
        "survival": {
            "embeddings_dir": Path("embeddings/os"),
            "df": os_eval,
            "label_col": "one_year_survival",
            "id_col": "Patient_mrn",
        },
        "mgmt": {
            "embeddings_dir": Path("embeddings/mgmt"),
            "df": mgmt_df,
            "label_col": "mgmt_label",
            "id_col": "mrn",
        },
    }

    for task_name, task_config in tasks.items():
        logger.info(f"\nEvaluating {task_name} classification...")
        results = {}

        embedding_files = list(task_config["embeddings_dir"].glob("*.npy"))

        for emb_file in embedding_files:
            embeddings, model_name = load_embeddings(emb_file)
            if embeddings is None:
                continue

            df = task_config["df"]
            # Filter out rows with null labels
            valid_mask = df[task_config["label_col"]].notna()
            labels = df[task_config["label_col"]][valid_mask].values
            embeddings = embeddings[valid_mask]

            logger.info(f"Evaluating {model_name}...")
            metrics = run_kfold_evaluation(embeddings, labels)
            results[model_name] = metrics

        # Save results
        results_table = format_results_table(results, task_name)
        output_file = results_dir / f"{task_name}_results.md"
        output_file.write_text(results_table)

        # Save raw metrics
        metrics_file = results_dir / f"{task_name}_metrics.json"
        with open(metrics_file, "w") as f:
            json.dump(results, f, indent=2)

        logger.info(f"Results saved to {output_file}")
        logger.info("\nResults Table:")
        logger.info("\n" + results_table)


if __name__ == "__main__":
    main()
